In [40]:
import numpy as np
from dipy.io.streamline import load_tractogram
import argparse
import pandas as pd
import os.path

import sys
sys.path.append('../script/')
from distances import distance_cores_svm
from from_trx_to_numpy import trx_to_numpy_tract_list

In [41]:
#HCP
subjects_HCP=['599469', '599671', '601127', '613538', '620434', '622236', '623844', '627549', '638049', '644044', '645551',
           '654754', '665254','672756','673455', '677968', '679568', '680957', '683256', '685058', '687163', '690152',
           '695768', '702133', '704238', '705341', '709551','715041','715647','729254', '729557', '732243', '734045', '742549',
           '748258', '748662', '749361', '751348', '753251', '756055', '759869', '761957','765056', '770352','771354', '779370',
           '782561', '784565', '786569', '789373', '792564', '792766', '802844', '814649', '816653', '826353','826454', '833148',
           '833249','837560', '837964', '845458', '849971', '856766', '857263', '859671', '861456', '865363', '871762', '871964',
           '872158', '872764', '877168', '877269','887373', '889579', '894673', '896778', '896879', '898176', '899885', '901038',
           '901139', '901442','904044', '907656', '910241', '912447', '917255','922854', '930449', '932554', '951457', '957974',
           '958976', '959574', '965367', '965771','978578', '979984', '983773', '984472', '987983', '991267','992774']

#PPMI/Control
subjects_Control=['156484','180966','187823','212409','232204','241154','253417','293209a','293209b','293727','305710','311373','315233',
                  '335195']
subjects_Control=['156484']

#PPMI/PD
subjects_PD=['100006','100007','100018','100267','100268','101026','101070','101146','101174','101279','101476','101479','101675',
             '101742','101751','101841','102053','102068a','102068b','102078','102305a','102305b','102321a','102321b','102978','106703',
             '107099','107648','109451','109619a','109619b','109910','110212','110220','111278','111383','111545a','111545b','112729',
             '113460','121109','121626','121830','126130','127075','127741','128196a','128196b','129515','33486a','133486b','133507a',
             '133507b','136600','136681a','136681b','137424a','137424b']
subjects_PD=['100267','100268']

#PPMI/Prodromal
subjects_Prodromal=['100122a','100122b','100122c','100250a','100250b','100971','101158a','101158b','101233a','101233b',
                    '101645a','101645b','101738','101761a','101761b','102073a','102073b','102119','102197a','102197b']

In [54]:
path="/home/gui/Documents/Post-doc/data/"

TRACT='CST_left'
alpha=0.0 #0.0 1e-05 5e-05 0.0001 0.0005 0.001 0.01 0.1 0.5 
nb_MAS=10

d1='PPMI/PD'
d2='PPMI/PD'

if d1>d2: #'HCP_105'<'PPMI/Control','PPMI/Control'<'PPMI/PD','PPMI/PD'<'PPMI/Prodromal'
    d1,d2=d2,d1

if d1 == 'HCP_105':
    subjects1=subjects_HCP
if d1 == 'PPMI/Control':
    subjects1=subjects_Control
if d1 == 'PPMI/PD':
    subjects1=subjects_PD
if d1 == 'PPMI/Prodromal':
    subjects1=subjects_Prodromal
    
if d2 == 'HCP_105':
    subjects2=subjects_HCP
if d2 == 'PPMI/Control':
    subjects2=subjects_Control
if d2 == 'PPMI/PD':
    subjects2=subjects_PD
if d2 == 'PPMI/Prodromal':
    subjects2=subjects_Prodromal

In [55]:
path1=path+str(d1)
path2=path+str(d2)

if "/" in d1:
    d1=d1.replace("/","_")
if "/" in d2:
    d2=d2.replace("/","_")

#Load all subject of the data
all_s1=pd.read_csv(path1+'/000000/all_subject.csv')["Subject ID"]
all_s2=pd.read_csv(path2+'/000000/all_subject.csv')["Subject ID"]

M = pd.DataFrame(np.zeros((len(all_s1),len(all_s2))), index=all_s1, columns=all_s2)

#If it does not exist yet save it, otherwise load it
if os.path.isfile(path+'matrix_distance/'+d1+'_'+d2+'_'+TRACT+'_L'+str(nb_MAS)+'_alpha'+str(alpha)+'.csv'):
    print('file already exist, loading it')
    M = pd.read_csv(path+'matrix_distance/'+d1+'_'+d2+'_'+TRACT+'_L'+str(nb_MAS)+'_alpha'+str(alpha)+'.csv',index_col=0)
else:  
    print('file does not exist writing it')
    M.to_csv(path+'matrix_distance/'+d1+'_'+d2+'_'+TRACT+'_L'+str(nb_MAS)+'_alpha'+str(alpha)+'.csv')
    M = pd.read_csv(path+'matrix_distance/'+d1+'_'+d2+'_'+TRACT+'_L'+str(nb_MAS)+'_alpha'+str(alpha)+'.csv',index_col=0)
M.index = M.index.astype(str)
M.columns = M.columns.astype(str)

file does not exist writing it


In [56]:
#Compute distance patients

if d1==d2:
    print('same dataset -> symmetric matrice')
    for i,s1 in enumerate(subjects1):
        print(s1,end=': ',flush=True)
        cores1 = load_tractogram(path1+"/"+s1+'/06-AugmentedTractsOT/7cores_MAS_'+TRACT+'_L'+str(nb_MAS)+'.trx', reference='same')
        m1,wI1,I1,w1,S1,labels_svm1,_=trx_to_numpy_tract_list(cores1,nb_MAS)
        for j,s2 in enumerate(subjects2):
            if int(s1)>int(s2):
                if M.at[s1, s2] != 0:
                    print('Overwritting a distance',end=' ',flush=True)
                cores2 = load_tractogram(path2+"/"+s2+'/06-AugmentedTractsOT/7cores_MAS_'+TRACT+'_L'+str(nb_MAS)+'.trx', reference='same')
                m2,wI2,I2,w2,S2,labels_svm2,_=trx_to_numpy_tract_list(cores2,nb_MAS)  
                cost,_,_,C_nb_pts=distance_cores_svm(m1,m2,wI1,wI2,I1,I2,w1,w2,S1,S2,labels_svm1,labels_svm2,alpha=alpha)
                M.at[s1, s2] = cost
                M.at[s2, s1] = cost
                #print(C_nb_pts.min())
        print()

else:
    for i,s1 in enumerate(subjects1):
        print(s1,end=': ',flush=True)
        cores1 = load_tractogram(path1+"/"+s1+'/06-AugmentedTractsOT/7cores_MAS_'+TRACT+'_L'+str(nb_MAS)+'.trx', reference='same')
        m1,wI1,I1,w1,S1,labels_svm1,_=trx_to_numpy_tract_list(cores1,nb_MAS)
        for j,s2 in enumerate(subjects2):
                if M.at[s1, s2] != 0:
                    print('Overwritting a distance',end=' ',flush=True)
                cores2 = load_tractogram(path2+"/"+s2+'/06-AugmentedTractsOT/7cores_MAS_'+TRACT+'_L'+str(nb_MAS)+'.trx', reference='same')
                m2,wI2,I2,w2,S2,labels_svm2,_=trx_to_numpy_tract_list(cores2,nb_MAS)  
                cost,_,_,C_nb_pts=distance_cores_svm(m1,m2,wI1,wI2,I1,I2,w1,w2,S1,S2,labels_svm1,labels_svm2,alpha=alpha)
                M.at[s1, s2] = cost
                #print(C_nb_pts.min())
        print()
    
M.to_csv(path+'matrix_distance/'+d1+'_'+d2+'_'+TRACT+'_L'+str(nb_MAS)+'_alpha'+str(alpha)+'.csv')

same dataset -> symmetric matrice
100267: 
100268: 

/home/gui/Documents/Post-doc/code/TractOT/.venv/lib/python3.12/site-packages/ot/lp/_network_simplex.py:574: UserWarning: Problem infeasible. Check that a and b are in the simplex
  check_result(result_code)


In [57]:
M.to_numpy()[M.to_numpy()!=0].min(),M.to_numpy()[M.to_numpy()!=0].max()

(np.float64(2.0037694916547477e-05), np.float64(2.0037694916547477e-05))